# Chuck Norris de la Tercera Edad

Fine-tuning de un LLM para que responda con "hechos" de Chuck Norris, pero en versión jubilado.

**El pipeline:**

1. Descargar chistes de la API gratuita de `chucknorris.io`
2. Filtrar los que no queremos que el modelo imite
3. Transformarlos a versión tercera edad (esto es lo que el modelo aprende)
4. Fine-tuning supervisado (SFT) con LoRA
5. Comparar el modelo antes y después

Cada sección define una clase con una responsabilidad. Ejecuta las celdas en orden.

In [ ]:
!pip install trl peft datasets transformers requests matplotlib --upgrade -q

## 1. Setup

In [ ]:
import hashlib
import json
import os
import random
import re
import time
from dataclasses import dataclass, field
from typing import List, Optional

print("Imports listos")

## 2. Configuración

Todos los parámetros del experimento en un solo sitio. Para probar otra cosa, cambia esto y vuelve a ejecutar.

Fíjate en `learning_rate`: es una *property*, no un valor fijo. Con LoRA se entrenan muy pocos parámetros, así que admite un learning rate unas 100 veces mayor que el fine-tuning completo. Atarlos juntos evita el error clásico de entrenar full FT con el LR de LoRA.

In [ ]:
@dataclass
class Config:
    # Datos
    size: int = 300
    min_len: int = 25
    max_len: int = 220
    test_ratio: float = 0.1
    cache_path: str = "chuck_raw.json"
    seed: int = 42
    # Modelo
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"
    max_new_tokens: int = 60
    temperature: float = 0.8
    # Entrenamiento
    epochs: int = 3
    batch_size: int = 8
    weight_decay: float = 0.01
    max_length: int = 256
    output_dir: str = "./chuck-senior-sft"
    # LoRA (None = fine-tuning completo)
    lora_rank: Optional[int] = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_targets: List[str] = field(
        default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj"]
    )

    @property
    def use_lora(self) -> bool:
        return self.lora_rank is not None

    @property
    def learning_rate(self) -> float:
        # Con LoRA se tocan pocos parámetros, así que admite un LR ~100x mayor.
        return 2e-4 if self.use_lora else 5e-6


config = Config()
print(f"LoRA: {config.use_lora} | learning rate: {config.learning_rate}")

## 3. Descargar chistes

La API `/random` repite muchísimo, así que deduplicamos y ponemos un tope de intentos. El resultado se cachea en disco: la segunda ejecución no vuelve a llamar a la API.

In [ ]:
class JokeFetcher:
    """Baja chistes de api.chucknorris.io y los cachea en disco."""

    URL = "https://api.chucknorris.io/jokes"

    def __init__(self, config: Config):
        self.config = config

    def get(self) -> List[str]:
        """Punto de entrada: caché si existe, si no descarga."""
        if os.path.exists(self.config.cache_path):
            return self._load_cache()
        jokes = self._download(int(self.config.size * 1.4))
        self._save_cache(jokes)
        return jokes

    def _load_cache(self) -> List[str]:
        with open(self.config.cache_path, encoding="utf-8") as f:
            jokes = json.load(f)
        print(f"Cargados {len(jokes)} chistes desde caché.")
        return jokes

    def _save_cache(self, jokes: List[str]) -> None:
        with open(self.config.cache_path, "w", encoding="utf-8") as f:
            json.dump(jokes, f, ensure_ascii=False, indent=2)
        print(f"Guardados {len(jokes)} chistes en {self.config.cache_path}")

    def _one_joke(self) -> Optional[str]:
        """Una llamada a /random con reintentos."""
        import requests

        for attempt in range(3):
            try:
                r = requests.get(f"{self.URL}/random", timeout=10)
                r.raise_for_status()
                return r.json().get("value", "").strip()
            except requests.RequestException as e:
                if attempt == 2:
                    print(f"[error] {e}")
                    return None
                time.sleep(2 ** attempt)
        return None

    def _download(self, n: int) -> List[str]:
        """La API repite mucho, así que deduplicamos con tope de intentos."""
        jokes, seen = [], set()

        for attempt in range(1, n * 6 + 1):
            if len(jokes) >= n:
                break
            joke = self._one_joke()
            if joke and joke.lower() not in seen:
                seen.add(joke.lower())
                jokes.append(joke)
            if attempt % 50 == 0:
                print(f"  ...{len(jokes)}/{n} recolectados")
            time.sleep(0.15)

        if len(jokes) < n:
            print(f"[aviso] solo {len(jokes)}/{n} chistes únicos")
        return jokes

Prueba rápida de que la API responde antes de descargar 400 chistes:

In [ ]:
fetcher = JokeFetcher(config)
print(fetcher._one_joke())

> **Sin internet?** Si la celda de arriba da error, salta la descarga y usa una lista fija. Descomenta y ejecuta esto, y el resto del notebook funciona igual:

```python
JOKES = [
    "Chuck Norris can divide by zero and still get a whole number every time",
    "Chuck Norris counted to infinity twice, then he did it backwards as well",
    # ...añade las tuyas
]
fetcher.get = lambda: JOKES
```

## 4. Filtrar

Los chistes de chucknorris.io los suben usuarios y hay material bastante feo. Como el modelo va a aprender a imitar exactamente lo que le demos, esto no es opcional.

In [ ]:
class JokeCleaner:
    """La API es contenido de usuarios: hay material que no queremos imitar."""

    BLOCKLIST = (
        "rape", "nigg", "fag", "aids", "cancer", "holocaust", "hitler",
        "porn", "rapist", "molest", "jew", "retard",
    )

    def __init__(self, config: Config):
        self.config = config

    def keep(self, joke: str) -> bool:
        return (
            self._good_length(joke)
            and self._no_bad_words(joke)
            and "http" not in joke.lower()
        )

    def clean(self, jokes: List[str]) -> List[str]:
        kept = [j for j in jokes if self.keep(j)]
        print(f"Chistes: {len(jokes)} descargados -> {len(kept)} tras filtrar")
        return kept

    def _good_length(self, joke: str) -> bool:
        return self.config.min_len <= len(joke) <= self.config.max_len

    def _no_bad_words(self, joke: str) -> bool:
        low = joke.lower()
        return not any(word in low for word in self.BLOCKLIST)

## 5. Convertir a versión jubilado

**Esta es la parte importante.** La API solo da el hecho original ("Chuck Norris can divide by zero"). El comportamiento que queremos enseñar lo fabricamos aquí: sustituciones léxicas más una coletilla senior.

La calidad de estas reglas *es* la calidad del fine-tuning. Si los resultados no te convencen, amplía `SUBSTITUTIONS` y `TAGS` antes que subir el número de épocas.

`_pick` elige por hash del chiste en vez de al azar, así el dataset es reproducible entre ejecuciones.

In [ ]:
class SeniorStyler:
    """Convierte un hecho de Chuck Norris en su versión tercera edad."""

    SUBSTITUTIONS = (
        ("roundhouse kick", "slow-motion roundhouse kick"),
        ("running", "power-walking"),
        ("runs", "power-walks"),
        ("ran", "power-walked"),
        ("sleeps", "naps"),
        ("punches", "gently taps"),
        ("punch", "gentle tap"),
        ("kills", "outlives"),
        ("kill", "outlive"),
        ("fights", "files a formal complaint against"),
        ("jumps", "carefully steps"),
        ("guns", "walking canes"),
        ("gun", "walking cane"),
        ("beer", "decaf"),
        ("coffee", "decaf"),
        ("car", "mobility scooter"),
        ("phone", "landline"),
        ("internet", "the Yellow Pages"),
        ("email", "handwritten letter"),
    )

    TAGS = (
        "But only between 3 and 5 PM, because after that it's bingo night.",
        "He does it every morning, right after his blood pressure pills.",
        "Then he sits down, because the knees aren't what they used to be.",
        "He'd do it more often, but the doctor said to take it easy.",
        "Afterwards he needs a 40-minute nap and a cup of decaf.",
        "He learned that in 1974 and has been complaining about it ever since.",
        "He can only do it once a week now, on Tuesdays, senior discount day.",
        "But he has to be home by 6, dinner is at 6 sharp.",
        "He'd explain how, but he left his reading glasses in the other room.",
        "It takes him a bit longer these days, but the result is the same.",
        "Then he tells the story about it. Twice. In the same afternoon.",
        "He wrote it down on a Post-it so he wouldn't forget.",
        "Not on cold mornings though, the hip acts up.",
        "He does it in slippers, because at his age comfort beats style.",
        "Right before his 4 PM appointment at the clinic.",
        "He says kids today couldn't do it, and he's probably right.",
        "Then he asks you to turn the volume down, it's too loud.",
        "He filed the paperwork for it in triplicate, just to be safe.",
        "Only after his physical therapy session, never on an empty stomach.",
    )

    PROMPTS = (
        "Tell me a Chuck Norris fact.",
        "Give me a fact about Chuck Norris.",
        "Chuck Norris fact, please.",
        "Say something impressive about Chuck Norris.",
        "What can Chuck Norris do?",
        "I need a Chuck Norris joke.",
        "Hit me with a Chuck Norris fact.",
        "Tell me something about Chuck Norris.",
    )

    def style(self, joke: str) -> str:
        """Original -> versión tercera edad."""
        text = self._substitute(joke)
        text = self._punctuate(text)
        return f"{text} {self._pick(self.TAGS, joke)}"

    def prompt_for(self, joke: str) -> str:
        return self._pick(self.PROMPTS, joke)

    def _substitute(self, text: str) -> str:
        for old, new in self.SUBSTITUTIONS:
            text = re.sub(rf"\b{re.escape(old)}\b", new, text, flags=re.IGNORECASE)
        return text

    def _punctuate(self, text: str) -> str:
        text = text.rstrip()
        return text if text.endswith((".", "!", "?")) else text + "."

    def _pick(self, options: tuple, key: str) -> str:
        """Determinista: mismo chiste, misma variante, siempre."""
        h = int(hashlib.md5(key.encode("utf-8")).hexdigest(), 16)
        return options[h % len(options)]

Antes de gastar GPU, mira qué produce la transformación:

In [ ]:
styler = SeniorStyler()

demo = [
    "Chuck Norris can divide by zero and still get a whole number every time",
    "Chuck Norris runs until the treadmill gets tired and asks for a break",
    "Chuck Norris counted to infinity twice, then he did it backwards as well",
]

for joke in demo:
    print(f"ANTES : {joke}")
    print(f"AHORA : {styler.style(joke)}\n")

## 6. Construir el dataset

`prompt` + `completion` es el formato que espera `SFTTrainer`. La columna `original` es solo para que puedas inspeccionar; se descarta antes de entrenar.

In [ ]:
class DatasetFactory:
    """Junta fetcher + cleaner + styler y devuelve un DatasetDict."""

    def __init__(self, config: Config):
        self.config = config
        self.fetcher = JokeFetcher(config)
        self.cleaner = JokeCleaner(config)
        self.styler = SeniorStyler()

    def make_rows(self) -> List[dict]:
        jokes = self.cleaner.clean(self.fetcher.get())[: self.config.size]
        return [
            {
                "prompt": self.styler.prompt_for(j),
                "completion": self.styler.style(j),
                "original": j,
            }
            for j in jokes
        ]

    def build(self):
        from datasets import Dataset, DatasetDict

        rows = self.make_rows()
        self.preview(rows)

        random.Random(self.config.seed).shuffle(rows)
        cut = int(len(rows) * (1 - self.config.test_ratio))

        data = DatasetDict({
            "train": Dataset.from_list(rows[:cut]),
            "test": Dataset.from_list(rows[cut:]),
        })
        print(f"Train: {len(data['train'])} | Test: {len(data['test'])}")
        return data

    def preview(self, rows: List[dict], n: int = 3) -> None:
        print("\n--- Ejemplos del dataset ---")
        for row in rows[:n]:
            print(f"\nORIGINAL   : {row['original']}")
            print(f"PROMPT     : {row['prompt']}")
            print(f"COMPLETION : {row['completion']}")

Ahora sí, descarga y construcción. La primera ejecución tarda un par de minutos por el `sleep` entre llamadas a la API.

In [ ]:
factory = DatasetFactory(config)
dataset = factory.build()
dataset

## 7. El modelo

`ChuckModel` envuelve modelo y tokenizer, y expone un único método útil: `ask()`.

In [ ]:
class ChuckModel:
    """Carga el modelo y genera respuestas."""

    def __init__(self, config: Config, model=None, tokenizer=None):
        self.config = config
        self.model = model
        self.tokenizer = tokenizer
        if model is None:
            self.load()

    def load(self) -> None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.tokenizer = AutoTokenizer.from_pretrained(self.config.model_name)
        self.tokenizer.pad_token = self.tokenizer.pad_token or self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            self.config.model_name, dtype=torch.bfloat16, device_map="auto"
        )
        self.model.generation_config.pad_token_id = self.tokenizer.pad_token_id

    def ask(self, prompt: str, verbose: bool = False) -> str:
        import torch

        with torch.inference_mode():
            ids = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            out = self.model.generate(
                input_ids=ids["input_ids"],
                attention_mask=ids["attention_mask"],
                max_new_tokens=self.config.max_new_tokens,
                do_sample=True,
                temperature=self.config.temperature,
                top_p=0.9,
                pad_token_id=self.tokenizer.pad_token_id,
            )

        text = self.tokenizer.decode(out[0], skip_special_tokens=True)
        answer = text[len(prompt):].strip() if text.startswith(prompt) else text.strip()
        if verbose:
            print(f"PROMPT : {prompt}\nOUTPUT : {answer}\n")
        return answer

## 8. Baseline: cómo responde ANTES

Guardamos estas respuestas para compararlas al final. Sin baseline no sabes si el fine-tuning hizo algo.

In [ ]:
class Reporter:
    """Curvas de loss y comparación antes/después."""

    PROMPTS = (
        "Tell me a Chuck Norris fact.",
        "What can Chuck Norris do?",
        "Hit me with a Chuck Norris fact.",
    )

    def __init__(self):
        self.before = {}

    def capture_before(self, chuck: "ChuckModel") -> None:
        print("\n===== ANTES DEL FINE-TUNING =====")
        for p in self.PROMPTS:
            self.before[p] = chuck.ask(p)
            print(f"{p}\n  -> {self.before[p]}\n")

    def compare(self, chuck: "ChuckModel") -> None:
        print("\n===== DESPUÉS DEL FINE-TUNING =====")
        for p in self.PROMPTS:
            print(f"{p}")
            print(f"  antes  : {self.before.get(p, '')[:120]}")
            print(f"  después: {chuck.ask(p)[:120]}")
            print("-" * 70)

    def plot(self, log_history) -> None:
        import matplotlib.pyplot as plt

        train = [l for l in log_history if "loss" in l and "eval_loss" not in l]
        evals = [l for l in log_history if "eval_loss" in l]

        plt.figure(figsize=(10, 6))
        plt.plot([l["step"] for l in train], [l["loss"] for l in train],
                 label="Training Loss", marker="o")
        plt.plot([l["step"] for l in evals], [l["eval_loss"] for l in evals],
                 label="Validation Loss", marker="s")
        plt.xlabel("Steps")
        plt.ylabel("Loss")
        plt.title("Training and Validation Loss")
        plt.legend()
        plt.grid(True)
        plt.show()

In [ ]:
baseline = ChuckModel(config)

reporter = Reporter()
reporter.capture_before(baseline)

Liberamos el modelo base de memoria, pero nos quedamos el tokenizer para reutilizarlo después:

In [ ]:
tokenizer = baseline.tokenizer

import gc, torch
del baseline
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("Memoria liberada")

## 9. Fine-tuning

`completion_only_loss=True` hace que la loss se calcule solo sobre la respuesta, no sobre el prompt: no queremos que el modelo aprenda a generar las preguntas.

Con LoRA el modelo base queda congelado y solo se entrenan matrices de rango bajo (`r=16`), escaladas por `alpha=32`.

In [ ]:
class Finetuner:
    """SFT con TRL. Usa LoRA si config.lora_rank no es None."""

    def __init__(self, config: Config):
        self.config = config
        self.trainer = None

    def train(self, dataset):
        from trl import SFTTrainer

        cols = ["prompt", "completion"]
        self.trainer = SFTTrainer(
            model=self.config.model_name,
            args=self._args(),
            train_dataset=dataset["train"].select_columns(cols),
            eval_dataset=dataset["test"].select_columns(cols),
            peft_config=self._peft(),
        )
        self.trainer.train()
        return self.trainer

    def _args(self):
        from trl import SFTConfig

        return SFTConfig(
            output_dir=self.config.output_dir,
            logging_strategy="steps",
            logging_steps=1,
            eval_strategy="steps",
            eval_steps=1,
            save_strategy="no",
            report_to="none",
            learning_rate=self.config.learning_rate,
            weight_decay=self.config.weight_decay,
            per_device_train_batch_size=self.config.batch_size,
            per_device_eval_batch_size=self.config.batch_size,
            num_train_epochs=self.config.epochs,
            max_length=self.config.max_length,
            completion_only_loss=True,  # loss solo sobre la respuesta
        )

    def _peft(self):
        if not self.config.use_lora:
            return None
        from peft import LoraConfig

        return LoraConfig(
            r=self.config.lora_rank,
            lora_alpha=self.config.lora_alpha,
            lora_dropout=self.config.lora_dropout,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=self.config.lora_targets,
        )

    @property
    def log_history(self):
        return self.trainer.state.log_history if self.trainer else []

    def save(self) -> str:
        path = f"{self.config.output_dir}/final"
        self.trainer.save_model(path)
        return path

In [ ]:
tuner = Finetuner(config)
tuner.train(dataset)

## 10. Resultados

Lo que buscas: ambas curvas bajando. Si la de validación sube mientras la de entrenamiento baja, estás sobreajustando: reduce épocas o sube el tamaño del dataset.

In [ ]:
reporter.plot(tuner.log_history)

In [ ]:
tuned = ChuckModel(config, tuner.trainer.model, tokenizer)
reporter.compare(tuned)

In [ ]:
print(f"Modelo guardado en {tuner.save()}")

## 11. Pruébalo tú

Con LoRA solo se guardan los adaptadores (unos pocos MB), no el modelo entero.

In [ ]:
tuned.ask("Tell me a Chuck Norris fact.", verbose=True)
tuned.ask("Chuck Norris fact, please.", verbose=True)

## Siguientes pasos

- **Mejorar el dataset**: más entradas en `SUBSTITUTIONS` y `TAGS` mejoran el resultado más que cualquier ajuste de hiperparámetros.
- **Probar hiperparámetros**: sube `lora_rank` a 32 o 64, o cambia `epochs`. Vuelve a ejecutar desde la celda de `Finetuner`.
- **Fine-tuning completo**: `config = Config(lora_rank=None)`. El learning rate se ajusta solo a 5e-6.
- **Modelo más grande**: `config = Config(model_name="Qwen/Qwen2.5-1.5B-Instruct")`, si la GPU aguanta.